# 마스킹 전후 생성 품질 평가

이 노트북은 새 목업 입력 파일 `recruiting_mock_dataset.csv`를 기준으로 같은 채용 데이터를 두 가지 경로로 실행한 뒤, 최종 리포트와 면접 질문지의 생성 품질 차이를 LLM judge로 비교합니다.

새 CSV는 아래 3개 입력 컬럼을 사용합니다.

- `company information`: 회사 정보 JSON 문자열
- `job_description`: 채용공고/JD JSON 문자열
- `resume`: 지원자 이력서 JSON 문자열

실행 흐름은 다음과 같습니다.

1. **일반 체인**: 원본 회사/JD와 Pinecone DB 검색 결과로 체크리스트를 생성합니다. 체크리스트 생성은 production 흐름과 맞춰 `extract_query -> Pinecone 검색 -> checklist 생성` 순서로 수행합니다. 이후 원본 회사/JD/이력서와 생성 체크리스트로 `analysis_graph.invoke()`를 실행합니다. 현재 분석 그래프는 내부에서 자기소개서 STAR 분석을 수행하므로 일반 체인도 STAR 분석을 포함합니다.
2. **마스킹 체인**: 로컬 Hugging Face 마스킹 모델로 회사/JD/이력서를 마스킹합니다. 마스킹된 회사/JD와 DB 검색 결과로 체크리스트를 생성하되, 검색된 DB 텍스트에도 같은 마스킹 맵을 적용합니다. 이후 마스킹된 입력으로 분석 그래프를 실행하고, 최종 산출물을 `unmask()`로 복호화합니다.
3. **LLM judge**: 두 결과를 같은 원본 입력과 각 체인에서 생성된 체크리스트를 기준으로 독립 채점한 뒤 paired delta를 계산합니다.

## 평가 지표와 루브릭

LLM judge는 아래 지표를 모두 1~5점으로 채점합니다. 5점이 가장 좋습니다.

- **source_fidelity**: 원본 이력서, 회사 정보, JD, 생성 체크리스트의 핵심 근거를 왜곡하지 않았는가.
- **coverage**: 직무 요구사항, 지원자 핵심 경험, 체크리스트 true/false 근거가 충분히 반영되었는가.
- **hallucination_control**: 원본에 없는 회사 내부 사정, 성과 수치, 경험, 기술 숙련도를 만들어내지 않았는가.
- **critical_information_retention**: 이름, 회사명, 학교, 경력, 기술, 도메인, 지원동기처럼 평가 결론에 중요한 정보가 누락되지 않았는가.
- **report_quality**: 등급, 요약, 강점, 우려, 체크포인트가 일관되고 채용 의사결정에 쓸 수 있는가.
- **question_quality**: 질문, 모범 답안, 질문 의도가 원본 근거에 기반하고 다양하며 면접 검증에 유용한가.
- **entity_recovery**: 마스킹 체인 산출물에서 복호화가 자연스럽게 되었고 `[COMP_NAME_1]` 같은 잔여 토큰이나 잘못 복원된 엔티티가 없는가.

`total_score`는 위 7개 지표의 산술 평균입니다. 핵심 비교값은 `masked.total_score - no_mask.total_score`입니다. 음수이면 마스킹 때문에 품질 손실이 생겼을 가능성이 있고, 양수이면 마스킹 체인이 더 안정적이거나 환각이 적었다는 의미입니다.

In [1]:
from __future__ import annotations

import contextlib
import csv
import inspect
import json
import os
import re
import sys
import time
import types
from copy import deepcopy
from pathlib import Path
from typing import Any, Literal

try:
    import pandas as pd
except ImportError:
    pd = None

from IPython.display import Markdown, display
from pydantic import BaseModel, Field

EVAL_DIR = Path.cwd()
if EVAL_DIR.name != "eval":
    EVAL_DIR = (Path.cwd() / "backend" / "common" / "eval").resolve()
else:
    EVAL_DIR = EVAL_DIR.resolve()

PROJECT_ROOT = EVAL_DIR.parents[2]
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from common import analysis_graph
from common import checklist_agent
from common.utils import load_env, mask, unmask

load_env()

DATA_PATH = EVAL_DIR / "recruiting_mock_dataset.csv"
CHAIN_CACHE_PATH = EVAL_DIR / "masking_quality_chain_outputs_recruiting_mock.json"
JUDGE_CACHE_PATH = EVAL_DIR / "masking_quality_judge_results_recruiting_mock.json"

COMPANY_COL = "company information"
JD_COL = "job_description"
RESUME_COL = "resume"
CHECKLIST_COUNT = int(os.getenv("MASKING_QUALITY_CHECKLIST_COUNT", "10"))
SAMPLE_SIZE = int(os.getenv("MASKING_QUALITY_SAMPLE_SIZE", "3"))
FORCE_REGENERATE = os.getenv("MASKING_QUALITY_FORCE_REGENERATE", "0").lower() in {"1", "true", "yes"}
FORCE_REJUDGE = os.getenv("MASKING_QUALITY_FORCE_REJUDGE", "0").lower() in {"1", "true", "yes"}

BASE_MODEL_NAME = os.getenv("MASKING_BASE_MODEL", "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")
ADAPTER_MODEL_NAME = os.getenv("MASKING_ADAPTER_MODEL", "dlfp22/exaone-masking-lora-best")
HF_TOKEN = os.getenv("HF_TOKEN")
USE_4BIT = os.getenv("MASKING_USE_4BIT", "1").lower() not in {"0", "false", "no"}
MASKING_MAX_NEW_TOKENS = int(os.getenv("MASKING_MAX_NEW_TOKENS", "512"))

JUDGE_MODEL = os.getenv("MASKING_QUALITY_JUDGE_MODEL", "gpt-4o-mini")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"DATA_PATH={DATA_PATH}")
print(f"SAMPLE_SIZE={SAMPLE_SIZE}")
print(f"CHECKLIST_COUNT={CHECKLIST_COUNT}")
print(f"MASKING_BASE_MODEL={BASE_MODEL_NAME}")
print(f"MASKING_ADAPTER_MODEL={ADAPTER_MODEL_NAME}")
print(f"JUDGE_MODEL={JUDGE_MODEL}")

PROJECT_ROOT=C:\project_skn\final\Final_project
DATA_PATH=C:\project_skn\final\Final_project\backend\common\eval\recruiting_mock_dataset.csv
SAMPLE_SIZE=3
CHECKLIST_COUNT=10
MASKING_BASE_MODEL=LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct
MASKING_ADAPTER_MODEL=dlfp22/exaone-masking-lora-best
JUDGE_MODEL=gpt-4o-mini


## 로컬 Hugging Face 마스킹 모델

아래 셀은 RunPod 핸들러와 같은 기본값을 사용합니다.

- base model: `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct`
- LoRA adapter: `dlfp22/exaone-masking-lora-best`

다른 모델을 시험하려면 환경변수 `MASKING_BASE_MODEL`, `MASKING_ADAPTER_MODEL`, `HF_TOKEN`, `MASKING_USE_4BIT`를 설정한 뒤 실행하면 됩니다.

In [2]:
LABEL_CATS = [
    "comp_name",
    "person_name",
    "address",
    "personal_info",
    "school_edu",
    "project_name",
    "jd_discrimination",
]

MASKING_SYSTEM_PROMPT = """
당신은 한국어 채용 데이터의 개인정보 및 민감 표현 마스킹 전문가입니다.
입력 JSON에서 마스킹이 필요한 원문 표현을 찾아 아래 7개 카테고리로 분류해 JSON object 하나만 반환하세요.

카테고리:
- comp_name: 회사명, 기관명, 고객사명, 이전 근무처명, 조직 식별명
- person_name: 지원자 본인, 교수, 추천인, 동료 등 사람 이름
- address: 주소, 출신지, 거주지
- personal_info: 연락처, 고유식별정보, 생년월일, 나이, 성별, 병역, 장애, 가족, 종교, 정치성향 등 민감 정보
- school_edu: 학교명, 교육기관명, 부트캠프명
- project_name: 내부 프로젝트명, 고객사 식별 가능 프로젝트명
- jd_discrimination: JD의 차별 소지 표현

규칙:
- 반드시 JSON object 하나만 출력합니다.
- 코드블록, 설명, 번역, 마크다운을 출력하지 않습니다.
- 입력에 등장한 원문 표현 그대로 추출합니다.
- 같은 표현은 한 번만 넣습니다.
- 해당 카테고리에 값이 없으면 빈 리스트를 넣습니다.
- 아래 7개 키 외 다른 키를 만들지 않습니다.

출력 형식:
{
  "comp_name": [],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": [],
  "project_name": [],
  "jd_discrimination": []
}
""".strip()


def empty_masking_result() -> dict[str, list[str]]:
    return {key: [] for key in LABEL_CATS}


def parse_prediction_json(text: str) -> dict[str, list[str]]:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    first = cleaned.find("{")
    last = cleaned.rfind("}")
    if first == -1 or last == -1 or first >= last:
        raise ValueError(f"JSON object를 찾지 못했습니다: {cleaned[:300]}")
    obj = json.loads(cleaned[first : last + 1])
    if not isinstance(obj, dict):
        return empty_masking_result()
    result = empty_masking_result()
    for key in LABEL_CATS:
        values = obj.get(key, [])
        if isinstance(values, list):
            result[key] = [str(value) for value in values if str(value).strip()]
    return result


def _patch_transformers_compat() -> None:
    try:
        import transformers.utils.generic as generic_utils

        if not hasattr(generic_utils, "maybe_autocast"):
            generic_utils.maybe_autocast = lambda *args, **kwargs: contextlib.nullcontext()
    except Exception as exc:
        print(f"[WARN] maybe_autocast patch skipped: {exc}")

    try:
        import transformers.modeling_rope_utils as rope_utils
        from typing import TypedDict

        if not hasattr(rope_utils, "RopeParameters"):
            class RopeParameters(TypedDict, total=False):
                rope_type: str
                factor: float
                low_freq_factor: float
                high_freq_factor: float
                original_max_position_embeddings: int
                attention_factor: float
                beta_fast: float
                beta_slow: float
                short_factor: list[float]
                long_factor: list[float]

            rope_utils.RopeParameters = RopeParameters
    except Exception as exc:
        print(f"[WARN] RopeParameters patch skipped: {exc}")

    try:
        import transformers.integrations as tf_integrations

        def noop_kernel_patch(*args: Any, **kwargs: Any):
            if args and callable(args[0]) and len(args) == 1:
                return args[0]

            def decorator(fn):
                return fn

            return decorator

        for name in ("use_kernel_forward_from_hub", "use_kernel_func_from_hub", "use_kernelized_func"):
            if not hasattr(tf_integrations, name):
                setattr(tf_integrations, name, noop_kernel_patch)
    except Exception as exc:
        print(f"[WARN] kernel integration patch skipped: {exc}")


def _patch_exaone_if_needed(model):
    if getattr(model, "_exaone_compat_patched", False):
        return model

    if hasattr(model, "transformer") and hasattr(model.transformer, "wte"):
        embed = model.transformer.wte
    elif hasattr(model, "transformer") and hasattr(model.transformer, "embed_tokens"):
        embed = model.transformer.embed_tokens
    elif hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
        embed = model.model.embed_tokens
    else:
        return model

    model.get_input_embeddings = lambda: embed
    model.set_input_embeddings = lambda value: setattr(embed, "weight", value.weight)

    try:
        import transformers.masking_utils as masking_utils

        original_create_causal_mask = masking_utils.create_causal_mask
        original_params = inspect.signature(original_create_causal_mask).parameters

        def create_causal_mask_compat(*args: Any, **kwargs: Any):
            if "input_embeds" in kwargs and "input_embeds" not in original_params:
                value = kwargs.pop("input_embeds")
                if "inputs_embeds" in original_params:
                    kwargs["inputs_embeds"] = value
                elif "input_tensor" in original_params:
                    kwargs["input_tensor"] = value
                else:
                    kwargs["input_ids"] = value
            elif "inputs_embeds" in kwargs and "inputs_embeds" not in original_params:
                value = kwargs.pop("inputs_embeds")
                if "input_embeds" in original_params:
                    kwargs["input_embeds"] = value
                elif "input_tensor" in original_params:
                    kwargs["input_tensor"] = value
                else:
                    kwargs["input_ids"] = value

            accepts_var_kwargs = any(
                param.kind == inspect.Parameter.VAR_KEYWORD for param in original_params.values()
            )
            if not accepts_var_kwargs:
                kwargs = {key: value for key, value in kwargs.items() if key in original_params}
            return original_create_causal_mask(*args, **kwargs)

        masking_utils.create_causal_mask = create_causal_mask_compat
        for module in list(sys.modules.values()):
            if module is not None and getattr(module, "create_causal_mask", None) is original_create_causal_mask:
                setattr(module, "create_causal_mask", create_causal_mask_compat)
    except Exception as exc:
        print(f"[WARN] EXAONE causal mask patch skipped: {exc}")

    backbone = getattr(model, "transformer", None) or getattr(model, "model", None)
    if backbone is not None and not hasattr(backbone, "_exaone_forward_patched"):
        original_forward = backbone.forward

        def patched_forward(self, *args: Any, **kwargs: Any):
            if "input_embeds" in kwargs and "inputs_embeds" not in kwargs:
                kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
            return original_forward(*args, **kwargs)

        backbone.forward = types.MethodType(patched_forward, backbone)
        backbone._exaone_forward_patched = True

    model._exaone_compat_patched = True
    return model


_masking_tokenizer = None
_masking_model = None


def get_local_masking_model():
    global _masking_tokenizer, _masking_model
    if _masking_tokenizer is not None and _masking_model is not None:
        return _masking_tokenizer, _masking_model

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    _patch_transformers_compat()
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model_kwargs: dict[str, Any] = {
        "token": HF_TOKEN,
        "trust_remote_code": True,
    }
    if torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"
        if USE_4BIT:
            compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=True,
            )
        else:
            model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, **model_kwargs)
    base_model = _patch_exaone_if_needed(base_model)
    model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_NAME, token=HF_TOKEN)
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    _masking_tokenizer = tokenizer
    _masking_model = model
    return tokenizer, model


def predict_masking(input_text: str, max_new_tokens: int = MASKING_MAX_NEW_TOKENS) -> str:
    import torch

    tokenizer, model = get_local_masking_model()
    messages = [
        {"role": "system", "content": MASKING_SYSTEM_PROMPT},
        {"role": "user", "content": input_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = inputs.to(next(model.parameters()).device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()


def invoke_local_hf_masking(data: dict[str, Any]) -> dict[str, Any]:
    input_text = json.dumps(data, ensure_ascii=False, indent=2)
    raw = predict_masking(input_text)
    parsed = parse_prediction_json(raw)
    return {"raw": raw, "result": parsed}


print("로컬 HF 마스킹 함수 준비 완료. 실제 모델 로드는 마스킹 체인을 처음 실행할 때 발생합니다.")

로컬 HF 마스킹 함수 준비 완료. 실제 모델 로드는 마스킹 체인을 처음 실행할 때 발생합니다.


## 새 Mock 데이터 로드와 체인 실행 함수

`recruiting_mock_dataset.csv`에는 골드 체크리스트/리포트/질문지가 없으므로, 두 체인 모두 입력 회사/JD와 DB 검색 결과를 이용해 체크리스트를 새로 생성합니다.

In [3]:
def parse_jsonish(value: Any, *, field_name: str = "") -> Any:
    if isinstance(value, (dict, list)):
        return value
    if value is None:
        raise ValueError(f"{field_name} is empty.")
    text = str(value).strip()
    if not text:
        raise ValueError(f"{field_name} is empty.")
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{field_name} must be a JSON string: {text[:200]}") from exc


def load_dataset(path: Path) -> list[dict[str, Any]]:
    if pd is not None:
        df = pd.read_csv(path, encoding="utf-8-sig")
        rows = df.to_dict(orient="records")
    else:
        with path.open(encoding="utf-8-sig", newline="") as f:
            rows = list(csv.DictReader(f))

    required = {COMPANY_COL, JD_COL, RESUME_COL}
    if not rows:
        raise ValueError(f"Dataset is empty: {path}")
    missing = required - set(rows[0].keys())
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return rows


def row_to_payload(row: dict[str, Any], index: int) -> dict[str, Any]:
    return {
        "set_id": int(row.get("set_id", index)),
        "company": parse_jsonish(row[COMPANY_COL], field_name=COMPANY_COL),
        "jd": parse_jsonish(row[JD_COL], field_name=JD_COL),
        "resume": parse_jsonish(row[RESUME_COL], field_name=RESUME_COL),
    }


def split_chain_result(result: dict[str, Any]) -> dict[str, Any]:
    questions = result.get("question") or result.get("questions") or []
    report = {key: value for key, value in result.items() if key not in {"question", "questions"}}
    return {"report": report, "questions": questions, "full_result": result}


def generate_checklist_no_mask(company: dict[str, Any], jd: dict[str, Any], cnt: int = CHECKLIST_COUNT) -> dict[str, Any]:
    query = checklist_agent.invoke_extract_query_node(compinfo=deepcopy(company), jdinfo=deepcopy(jd))
    db_data = checklist_agent.invoke_search_embedding_node(query=query, cnt=cnt)
    checklist = checklist_agent.invoke_fit_checklist_node(
        company_info=deepcopy(company),
        jd_info=deepcopy(jd),
        db_data=deepcopy(db_data),
        checklist_count=cnt,
    )
    return {"query": query, "db_data": db_data, "checklist": checklist}


def generate_checklist_masked(
    masked_company: dict[str, Any],
    masked_jd: dict[str, Any],
    mask_result: dict[str, Any],
    cnt: int = CHECKLIST_COUNT,
) -> dict[str, Any]:
    query = checklist_agent.invoke_extract_query_node(compinfo=deepcopy(masked_company), jdinfo=deepcopy(masked_jd))
    db_data = checklist_agent.invoke_search_embedding_node(query=query, cnt=cnt)
    masked_db_data = mask({"db_data": deepcopy(db_data)}, mask_result)["db_data"]
    checklist = checklist_agent.invoke_fit_checklist_node(
        company_info=deepcopy(masked_company),
        jd_info=deepcopy(masked_jd),
        db_data=deepcopy(masked_db_data),
        checklist_count=cnt,
    )
    return {"query": query, "db_data": db_data, "masked_db_data": masked_db_data, "checklist": checklist}


def run_no_mask_chain(payload: dict[str, Any]) -> dict[str, Any]:
    checklist_bundle = generate_checklist_no_mask(payload["company"], payload["jd"])
    result = analysis_graph.invoke(
        company_dict=deepcopy(payload["company"]),
        jd_dict=deepcopy(payload["jd"]),
        checklist=deepcopy(checklist_bundle["checklist"]),
        resume_dict=deepcopy(payload["resume"]),
    )
    split_result = split_chain_result(result)
    split_result["checklist_generation"] = checklist_bundle
    return split_result


def run_masked_chain(payload: dict[str, Any]) -> dict[str, Any]:
    source_input = {
        "company": deepcopy(payload["company"]),
        "jd": deepcopy(payload["jd"]),
        "resume": deepcopy(payload["resume"]),
    }
    masking_output = invoke_local_hf_masking(source_input)
    mask_result = masking_output["result"]
    masked_input = mask(deepcopy(source_input), mask_result)

    checklist_bundle = generate_checklist_masked(
        masked_company=deepcopy(masked_input["company"]),
        masked_jd=deepcopy(masked_input["jd"]),
        mask_result=mask_result,
    )

    masked_chain_result = analysis_graph.invoke(
        company_dict=deepcopy(masked_input["company"]),
        jd_dict=deepcopy(masked_input["jd"]),
        checklist=deepcopy(checklist_bundle["checklist"]),
        resume_dict=deepcopy(masked_input["resume"]),
    )
    unmasked_result = unmask(deepcopy(masked_chain_result), mask_result)
    split_result = split_chain_result(unmasked_result)
    split_result["mask_result"] = mask_result
    split_result["masking_raw"] = masking_output["raw"]
    split_result["masked_input"] = masked_input
    split_result["checklist_generation"] = checklist_bundle
    split_result["unmasked_checklist_generation"] = unmask(deepcopy(checklist_bundle), mask_result)
    split_result["masked_raw_result"] = masked_chain_result
    return split_result


dataset_rows = load_dataset(DATA_PATH)
samples = [row_to_payload(row, index) for index, row in enumerate(dataset_rows[:SAMPLE_SIZE])]
summary_rows = [
    {
        "set_id": sample["set_id"],
        "company_name": sample["company"].get("company_name", ""),
        "job_name": sample["jd"].get("job_name", ""),
        "resume_name": sample["resume"].get("name", ""),
    }
    for sample in samples
]
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    display(summary_rows)

,set_id,company_name,job_name,resume_name
0,0,그린모빌리티,백엔드 개발자 채용,백수민
1,1,데이터브릿지,데이터 엔지니어 채용,정하람
2,2,코드윈드,프론트엔드 개발자 채용,오건우


## 1. 일반 체인 실행, 2. 마스킹 체인 실행

이 셀은 체크리스트 생성 과정에서 OpenAI embedding, Pinecone DB 검색, LLM 생성을 사용하므로 비용과 시간이 큽니다. 이미 실행한 결과가 있으면 `masking_quality_chain_outputs_recruiting_mock.json` 캐시를 재사용합니다. 다시 생성하려면 환경변수 `MASKING_QUALITY_FORCE_REGENERATE=1`을 설정하세요.

In [4]:
def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


if CHAIN_CACHE_PATH.exists() and not FORCE_REGENERATE:
    chain_records = load_json(CHAIN_CACHE_PATH)
    print(f"Loaded cache: {CHAIN_CACHE_PATH} ({len(chain_records)} records)")
else:
    chain_records = []
    for payload in samples:
        sid = payload["set_id"]
        started_at = time.time()
        print(f"[set_id={sid}] Start no-mask chain: original company/JD + DB checklist generation")
        no_mask_output = run_no_mask_chain(payload)
        print(f"[set_id={sid}] Start masked chain: local HF masking + masked company/JD + DB checklist generation")
        masked_output = run_masked_chain(payload)
        elapsed = round(time.time() - started_at, 2)
        chain_records.append(
            {
                "set_id": sid,
                "source": {
                    "company": payload["company"],
                    "jd": payload["jd"],
                    "resume": payload["resume"],
                },
                "no_mask": no_mask_output,
                "masked": masked_output,
                "elapsed_sec": elapsed,
            }
        )
        save_json(CHAIN_CACHE_PATH, chain_records)
        print(f"[set_id={sid}] Done: {elapsed} sec")

print(f"Prepared {len(chain_records)} records")

[set_id=0] Start no-mask chain: original company/JD + DB checklist generation


KeyboardInterrupt: 

## 산출물 확인

각 `set_id`마다 일반 체인 결과를 먼저 보고, 이어서 마스킹 후 복호화된 체인 결과를 봅니다. 체크리스트도 각 체인에서 새로 생성된 값을 함께 확인합니다.

In [ ]:
def find_mask_tokens(obj: Any) -> list[str]:
    text = json.dumps(obj, ensure_ascii=False)
    return sorted(set(re.findall(r"\[[A-Z_]+_\d+\]", text)))


def display_table(rows: Any) -> None:
    if pd is not None and isinstance(rows, list):
        display(pd.DataFrame(rows))
    else:
        display(rows)


def display_chain_output(record: dict[str, Any]) -> None:
    sid = record["set_id"]
    display(Markdown(f"## set_id={sid}"))

    display(Markdown("### No-mask chain: checklist generated, STAR analysis included"))
    display(Markdown("#### Checklist generation"))
    display(record["no_mask"]["checklist_generation"])
    display(Markdown("#### Report"))
    display(record["no_mask"]["report"])
    display(Markdown("#### Interview questions"))
    display_table(record["no_mask"]["questions"])

    display(Markdown("### Masked chain: local HF masking, checklist generated, STAR analysis included, then unmasked"))
    display(Markdown("#### Masking result"))
    display(record["masked"]["mask_result"])
    display(Markdown("#### Checklist generation, displayed after unmasking"))
    display(record["masked"].get("unmasked_checklist_generation", record["masked"]["checklist_generation"]))
    unresolved_tokens = find_mask_tokens(record["masked"]["full_result"])
    display(Markdown(f"Unresolved mask tokens: `{unresolved_tokens}`"))
    display(Markdown("#### Report"))
    display(record["masked"]["report"])
    display(Markdown("#### Interview questions"))
    display_table(record["masked"]["questions"])


for record in chain_records:
    display_chain_output(record)

## LLM judge 스키마와 프롬프트

judge는 두 결과를 서로 직접 베끼듯 비교하지 않고, 같은 원본 입력에 대한 독립 산출물로 먼저 채점합니다. 새 데이터셋에는 골드 리포트/질문지가 없으므로, 원본 입력과 각 체인에서 생성한 체크리스트를 기준 근거로 삼습니다.

In [ ]:
class ChainQualityScore(BaseModel):
    source_fidelity: int = Field(ge=1, le=5, description="원본 근거 보존과 왜곡 방지")
    coverage: int = Field(ge=1, le=5, description="핵심 정보와 체크리스트 근거 포함 정도")
    hallucination_control: int = Field(ge=1, le=5, description="원본에 없는 내용 생성 억제")
    critical_information_retention: int = Field(ge=1, le=5, description="중요 정보 손실 방지")
    report_quality: int = Field(ge=1, le=5, description="최종 리포트 품질")
    question_quality: int = Field(ge=1, le=5, description="면접 질문지 품질")
    entity_recovery: int = Field(ge=1, le=5, description="엔티티 표현 자연스러움과 복호화 품질")
    total_score: float = Field(ge=1, le=5, description="위 7개 지표 평균")
    major_losses: list[str] = Field(default_factory=list, description="중요 내용 손실 사례")
    hallucinations: list[str] = Field(default_factory=list, description="원본에 없는 내용 생성 사례")
    rationale: str = Field(description="점수 근거 요약")


class PairQualityJudgement(BaseModel):
    set_id: int
    no_mask: ChainQualityScore
    masked: ChainQualityScore
    comparative_preference: Literal["no_mask_better", "masked_better", "tie"]
    masking_delta_summary: str = Field(description="마스킹 체인의 품질 차이 요약")
    unresolved_mask_tokens: list[str] = Field(default_factory=list, description="복호화 후 남은 마스킹 토큰")
    final_recommendation: str = Field(description="마스킹 체인 사용 여부와 보완점")


JUDGE_SYSTEM_PROMPT = """
당신은 채용 분석 리포트와 면접 질문지의 생성 품질을 평가하는 엄격한 LLM judge입니다.
목표는 마스킹이 적용된 체인이 마스킹 없는 체인 대비 핵심 정보 손실, 환각, 복호화 오류를 일으키는지 평가하는 것입니다.

채점 기준은 모두 1~5점입니다.
5점: 원본 입력과 생성 체크리스트에 충실하며 채용 검토에 바로 사용 가능
4점: 사소한 누락은 있으나 핵심 판단에는 문제 없음
3점: 일부 핵심 근거가 약하거나 질문/리포트 중 하나의 품질이 불안정
2점: 중요한 내용 손실, 부정확한 일반화, 근거 없는 문장이 여러 개 있음
1점: 원본과 의미가 크게 다르거나 채용 판단에 쓰기 어려움

반드시 원본 입력과 각 체인에서 생성한 체크리스트에 있는 정보만 근거로 판단하세요.
두 체인은 체크리스트도 각각 생성하므로, 체크리스트 자체의 품질 차이가 리포트/질문지 품질에 영향을 준 경우 그 점을 rationale에 명시하세요.
마스킹 체인은 복호화된 최종 결과를 평가하되, 잔여 마스킹 토큰이나 잘못 복원된 엔티티가 있으면 entity_recovery와 total_score를 낮추세요.
total_score는 source_fidelity, coverage, hallucination_control, critical_information_retention, report_quality, question_quality, entity_recovery의 산술 평균으로 계산하세요.
근거, 요약, 손실 사례, 환각 사례, 권고사항은 모두 한국어로 작성하세요.
""".strip()


def compact_json(obj: Any, max_chars: int = 24000) -> str:
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...TRUNCATED..."


def build_judge_user_prompt(record: dict[str, Any]) -> str:
    payload = {
        "set_id": record["set_id"],
        "source_input": record["source"],
        "no_mask_output": {
            "generated_checklist": record["no_mask"]["checklist_generation"].get("checklist", []),
            "report": record["no_mask"]["report"],
            "questions": record["no_mask"]["questions"],
        },
        "masked_then_unmasked_output": {
            "mask_result": record["masked"].get("mask_result", {}),
            "generated_checklist_after_unmask": record["masked"].get("unmasked_checklist_generation", {}).get("checklist", []),
            "report": record["masked"]["report"],
            "questions": record["masked"]["questions"],
            "unresolved_mask_tokens_detected_by_regex": find_mask_tokens(record["masked"]["full_result"]),
        },
    }
    return "다음 JSON을 평가하세요.\n\n" + compact_json(payload)


def call_judge(record: dict[str, Any]) -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": build_judge_user_prompt(record)},
    ]

    parse_method = getattr(client.beta.chat.completions, "parse", None)
    if parse_method:
        response = parse_method(
            model=JUDGE_MODEL,
            messages=messages,
            response_format=PairQualityJudgement,
            temperature=0,
        )
        return response.choices[0].message.parsed.model_dump()

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0,
    )
    return PairQualityJudgement.model_validate_json(response.choices[0].message.content).model_dump()


print("LLM judge 준비 완료")

## 3. LLM judge 실행

In [ ]:
def flatten_dict(obj: dict[str, Any], prefix: str = "") -> dict[str, Any]:
    flattened = {}
    for key, value in obj.items():
        path = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            flattened.update(flatten_dict(value, path))
        else:
            flattened[path] = value
    return flattened


if JUDGE_CACHE_PATH.exists() and not FORCE_REJUDGE:
    judge_records = load_json(JUDGE_CACHE_PATH)
    print(f"Loaded judge cache: {JUDGE_CACHE_PATH} ({len(judge_records)} records)")
else:
    judge_records = []
    for record in chain_records:
        sid = record["set_id"]
        print(f"[set_id={sid}] Run LLM judge")
        judgement = call_judge(record)
        judge_records.append(judgement)
        save_json(JUDGE_CACHE_PATH, judge_records)
    print(f"Judge complete: {len(judge_records)} records")

judge_rows = [flatten_dict(record) for record in judge_records]
if pd is not None:
    judge_df = pd.DataFrame(judge_rows)
    display(judge_df)
else:
    judge_df = judge_rows
    display(judge_rows)

## 4. 최종 수치 비교

아래 표의 `delta_masked_minus_no_mask`가 핵심 비교값입니다. 음수이면 마스킹 체인 품질이 낮아진 지표이고, 양수이면 마스킹 체인이 더 나은 지표입니다.

In [ ]:
METRICS = [
    "source_fidelity",
    "coverage",
    "hallucination_control",
    "critical_information_retention",
    "report_quality",
    "question_quality",
    "entity_recovery",
    "total_score",
]


def col_values(rows: list[dict[str, Any]], col: str) -> list[float]:
    values = []
    for row in rows:
        value = row.get(col)
        if value is not None:
            values.append(float(value))
    return values


def mean(values: list[float]) -> float | None:
    return sum(values) / len(values) if values else None


rows_for_summary = judge_df.to_dict(orient="records") if pd is not None else judge_df
summary_rows = []
for metric in METRICS:
    no_col = f"no_mask.{metric}"
    masked_col = f"masked.{metric}"
    no_values = col_values(rows_for_summary, no_col)
    masked_values = col_values(rows_for_summary, masked_col)
    no_avg = mean(no_values)
    masked_avg = mean(masked_values)
    summary_rows.append(
        {
            "metric": metric,
            "no_mask_avg": no_avg,
            "masked_avg": masked_avg,
            "delta_masked_minus_no_mask": None if no_avg is None or masked_avg is None else masked_avg - no_avg,
            "no_mask_min": min(no_values) if no_values else None,
            "masked_min": min(masked_values) if masked_values else None,
        }
    )

pairwise_rows = []
for row in rows_for_summary:
    no_total = row.get("no_mask.total_score")
    masked_total = row.get("masked.total_score")
    pairwise_rows.append(
        {
            "set_id": row.get("set_id"),
            "comparative_preference": row.get("comparative_preference"),
            "masking_delta_summary": row.get("masking_delta_summary"),
            "final_recommendation": row.get("final_recommendation"),
            "no_mask_total": no_total,
            "masked_total": masked_total,
            "total_delta": None if no_total is None or masked_total is None else float(masked_total) - float(no_total),
        }
    )

preference_counts = {}
for row in rows_for_summary:
    key = row.get("comparative_preference")
    preference_counts[key] = preference_counts.get(key, 0) + 1
preference_rows = [{"preference": key, "count": value} for key, value in preference_counts.items()]

if pd is not None:
    summary_df = pd.DataFrame(summary_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)
    preference_df = pd.DataFrame(preference_rows)
    display(summary_df)
    display(pairwise_df)
    display(preference_df)
    summary_df.to_csv(EVAL_DIR / "masking_quality_metric_summary.csv", index=False, encoding="utf-8")
    pairwise_df.to_csv(EVAL_DIR / "masking_quality_pairwise_summary.csv", index=False, encoding="utf-8")
    pd.DataFrame(rows_for_summary).to_csv(EVAL_DIR / "masking_quality_judge_detail.csv", index=False, encoding="utf-8")
else:
    display(summary_rows)
    display(pairwise_rows)
    display(preference_rows)
    with (EVAL_DIR / "masking_quality_metric_summary.csv").open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)
    with (EVAL_DIR / "masking_quality_pairwise_summary.csv").open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(pairwise_rows[0].keys()))
        writer.writeheader()
        writer.writerows(pairwise_rows)
    with (EVAL_DIR / "masking_quality_judge_detail.csv").open("w", encoding="utf-8", newline="") as f:
        fieldnames = sorted({key for row in rows_for_summary for key in row.keys()})
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows_for_summary)

print("Saved:")
print(EVAL_DIR / "masking_quality_metric_summary.csv")
print(EVAL_DIR / "masking_quality_pairwise_summary.csv")
print(EVAL_DIR / "masking_quality_judge_detail.csv")